# Feature Engineering — Extended Dataset (2023 Q1 – 2025 Q4)

**Purpose:** Transform the cleaned 3-year panel dataset into a richer analytical dataset  
by creating time-based features that capture *change* and *dynamics*, not just static snapshots.

**Input:** `datasets/extended/gdp_unemployment_merged_extended.csv`  
**Output:** `datasets/extended/gdp_unemployment_featured.csv`

---
### Why feature engineering matters here

The original dataset had only 4 quarters of data (2025 only).  
The Pearson correlation was **−0.11 (p = 0.49)** — not statistically significant.  
The main reason was not a weak relationship: it was **too few observations and no time signal**.

With 12 quarters (2023–2025), we can now compute features like:
- **Quarter-over-quarter growth rates** — how fast is each economy changing?
- **Lagged variables** — does last quarter's GDP predict this quarter's unemployment?
- **Rolling averages** — what is the underlying trend, ignoring seasonal noise?
- **Encoded categoricals** — machine-learning-ready columns

These features are what turn raw data into something a model can actually learn from.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = pd.read_csv('datasets/extended/gdp_unemployment_merged_extended.csv', sep=';')

# Sort by country then quarter — required for all lag/diff operations
df['QUARTER'] = df['QUARTER'].astype(str)
df = df.sort_values(['REF_AREA', 'QUARTER']).reset_index(drop=True)

print('Shape:', df.shape)
print('Countries:', df['REF_AREA'].nunique())
print('Quarters:', sorted(df['QUARTER'].unique()))
print()
print(df.head(8).to_string())

## 1. Quarter-over-Quarter Growth Rates

**What:** Percentage change from one quarter to the next, computed *within each country*.  
**Why:** Okun's Law operates on *changes*, not levels.  
A country with GDP = \$70,000 that is shrinking may behave very differently from  
one with GDP = \$40,000 that is growing fast.

In [ ]:
def pct_change_within_country(series, group_col='REF_AREA'):
    """Percentage change within each country group."""
    return df.groupby(group_col)[series.name].pct_change() * 100

df['GDP_GROWTH_QOQ_PCT'] = df.groupby('REF_AREA')['GDP_PER_CAPITA_USD_PPP'].pct_change() * 100
df['UNE_CHANGE_QOQ_PP']  = df.groupby('REF_AREA')['UNE_RATE_PCT'].diff()

print('GDP_GROWTH_QOQ_PCT — quarter-over-quarter GDP growth (%)')
print('UNE_CHANGE_QOQ_PP  — quarter-over-quarter unemployment change (percentage points)')
print()
print('Sample (AUS):')
print(df[df['REF_AREA']=='AUS'][['REF_AREA','QUARTER','GDP_PER_CAPITA_USD_PPP','GDP_GROWTH_QOQ_PCT','UNE_RATE_PCT','UNE_CHANGE_QOQ_PP']].to_string())
print()
print(f'Rows with NaN in growth features (first quarter per country, expected): {df["GDP_GROWTH_QOQ_PCT"].isna().sum()}')

## 2. Lagged Variables

**What:** Value of GDP or unemployment from the *previous* quarter.  
**Why:** Economic effects take time. If a company sees falling GDP, it will cut jobs  
in the *next* quarter — not immediately. A 1-quarter lag captures this delay.  
This is also essential for any predictive model.

In [ ]:
df['GDP_LAG_1Q']  = df.groupby('REF_AREA')['GDP_PER_CAPITA_USD_PPP'].shift(1)
df['UNE_LAG_1Q']  = df.groupby('REF_AREA')['UNE_RATE_PCT'].shift(1)
df['GDP_LAG_2Q']  = df.groupby('REF_AREA')['GDP_PER_CAPITA_USD_PPP'].shift(2)
df['UNE_LAG_2Q']  = df.groupby('REF_AREA')['UNE_RATE_PCT'].shift(2)

print('Lag features added.')
print('Sample (DEU):')
print(df[df['REF_AREA']=='DEU'][['REF_AREA','QUARTER','GDP_PER_CAPITA_USD_PPP','GDP_LAG_1Q','UNE_RATE_PCT','UNE_LAG_1Q']].to_string())

## 3. Rolling Averages (Trend Smoothing)

**What:** Moving average over the last 4 quarters (1 full year).  
**Why:** Quarterly data has seasonal noise. A 4-quarter rolling average shows the  
underlying trend and makes year-over-year comparisons more stable.

In [ ]:
df['GDP_ROLLING_4Q']  = df.groupby('REF_AREA')['GDP_PER_CAPITA_USD_PPP'].transform(
    lambda x: x.rolling(window=4, min_periods=2).mean()
)
df['UNE_ROLLING_4Q']  = df.groupby('REF_AREA')['UNE_RATE_PCT'].transform(
    lambda x: x.rolling(window=4, min_periods=2).mean()
)

print('Rolling averages added (4-quarter window, min 2 quarters required).')
print('Sample (USA):')
print(df[df['REF_AREA']=='USA'][['REF_AREA','QUARTER','GDP_PER_CAPITA_USD_PPP','GDP_ROLLING_4Q','UNE_RATE_PCT','UNE_ROLLING_4Q']].to_string())

## 4. Year-over-Year Change

**What:** Change compared to the *same quarter one year ago* (4 quarters back).  
**Why:** Removes seasonal effects completely. If Q2-2025 was better than Q2-2024,  
that is real structural improvement — not just a seasonal bounce.

In [ ]:
df['GDP_GROWTH_YOY_PCT'] = df.groupby('REF_AREA')['GDP_PER_CAPITA_USD_PPP'].pct_change(periods=4) * 100
df['UNE_CHANGE_YOY_PP']  = df.groupby('REF_AREA')['UNE_RATE_PCT'].diff(periods=4)

print('Year-over-year features added.')
print('Sample (FRA):')
print(df[df['REF_AREA']=='FRA'][['REF_AREA','QUARTER','GDP_GROWTH_YOY_PCT','UNE_CHANGE_YOY_PP']].to_string())

## 5. Categorical Encoding

**What:** Convert text columns to numbers that a machine learning algorithm can use.  
**Why:** Models like k-Means, regression, and neural networks cannot work with strings.

In [ ]:
# 5.1 OBS_STATUS: ordinal encoding (A=final=2, P=provisional=1, E=estimated=0)
status_map = {'A': 2, 'P': 1, 'E': 0}
df['OBS_STATUS_ENC'] = df['OBS_STATUS'].map(status_map).fillna(1).astype(int)

# 5.2 QUARTER: numeric index within each country for ordered time dimension
quarter_order = sorted(df['QUARTER'].unique())
quarter_idx   = {q: i for i, q in enumerate(quarter_order)}
df['QUARTER_IDX'] = df['QUARTER'].map(quarter_idx)

# 5.3 YEAR: extracted from quarter string for grouping
df['YEAR'] = df['QUARTER'].str[:4].astype(int)

# 5.4 REF_AREA: label-encoded for ML (stored separately to keep country code readable)
country_codes = sorted(df['REF_AREA'].unique())
country_idx   = {c: i for i, c in enumerate(country_codes)}
df['COUNTRY_IDX'] = df['REF_AREA'].map(country_idx)

print('Encoded columns added:')
print('  OBS_STATUS_ENC — 2=final, 1=provisional, 0=estimated')
print('  QUARTER_IDX    — 0 to', df['QUARTER_IDX'].max(), '(chronological index)')
print('  YEAR           — 2023, 2024, or 2025')
print('  COUNTRY_IDX    — 0 to', df['COUNTRY_IDX'].max())
print()
print(df[['REF_AREA','QUARTER','OBS_STATUS','OBS_STATUS_ENC','QUARTER_IDX','YEAR','COUNTRY_IDX']].head(8).to_string())

## 6. GDP Group Label (High / Low)

**What:** Binary label splitting countries at the median GDP.  
**Why:** Useful as a target variable for classification models and for the hypothesis test  
already done in `unit3-assignment2.ipynb` — now computed across all 12 quarters.

In [ ]:
# Use the annual average GDP per country as the split criterion (avoids quarterly noise)
annual_avg = df.groupby('REF_AREA')['GDP_PER_CAPITA_USD_PPP'].mean()
gdp_median = annual_avg.median()
high_gdp_countries = set(annual_avg[annual_avg >= gdp_median].index)

df['GDP_GROUP'] = df['REF_AREA'].apply(lambda c: 'HIGH' if c in high_gdp_countries else 'LOW')
df['GDP_GROUP_ENC'] = (df['GDP_GROUP'] == 'HIGH').astype(int)   # 1=HIGH, 0=LOW

print(f'Median annual GDP (used as threshold): ${gdp_median:,.0f}')
print(f'HIGH-GDP countries: {len(high_gdp_countries)}')
print(f'LOW-GDP  countries: {df["REF_AREA"].nunique() - len(high_gdp_countries)}')
print()
print('HIGH-GDP group:', sorted(high_gdp_countries))

## 7. Final Dataset Overview

In [ ]:
print('=== Final featured dataset ===')
print(f'Shape: {df.shape}')
print(f'Countries: {df["REF_AREA"].nunique()}')
print(f'Quarters: {df["QUARTER"].nunique()} — from {df["QUARTER"].min()} to {df["QUARTER"].max()}')
print()
print('Columns:')
for col in df.columns:
    n_null = df[col].isna().sum()
    note = f'  ← {n_null} NaN (first quarters per country — expected)' if n_null > 0 else ''
    print(f'  {col:<30} dtype={str(df[col].dtype):<10}{note}')

print()
print('Descriptive statistics (key columns):')
print(df[['GDP_PER_CAPITA_USD_PPP','GDP_GROWTH_QOQ_PCT','GDP_GROWTH_YOY_PCT',
          'UNE_RATE_PCT','UNE_CHANGE_QOQ_PP','UNE_CHANGE_YOY_PP']].describe().round(3).to_string())

## 8. Correlation — Does More Data Change the Result?

Re-runs the Pearson and Spearman correlation from `unit3-assignment2.ipynb`  
but now using all 12 quarters (annual average per country) to see if the  
relationship becomes clearer or more significant.

In [ ]:
from scipy import stats

# Annual average per country over all 3 years
annual = df.groupby('REF_AREA')[['GDP_PER_CAPITA_USD_PPP', 'UNE_RATE_PCT']].mean()

r_p, p_p = stats.pearsonr(annual['GDP_PER_CAPITA_USD_PPP'], annual['UNE_RATE_PCT'])
r_s, p_s = stats.spearmanr(annual['GDP_PER_CAPITA_USD_PPP'], annual['UNE_RATE_PCT'])

print('=== Correlation on 3-year annual averages (n =', len(annual), 'countries) ===')
print(f'Pearson  r = {r_p:.4f}   p = {p_p:.4f}')
print(f'Spearman ρ = {r_s:.4f}   p = {p_s:.4f}')
print()

# Also: correlation on the YoY change features (Okun's Law proper test)
yoy = df[['GDP_GROWTH_YOY_PCT', 'UNE_CHANGE_YOY_PP']].dropna()
r_okun, p_okun = stats.pearsonr(yoy['GDP_GROWTH_YOY_PCT'], yoy['UNE_CHANGE_YOY_PP'])
print('=== Okun\'s Law test — YoY GDP growth vs YoY unemployment change ===')
print(f'Pearson  r = {r_okun:.4f}   p = {p_okun:.4f}   n = {len(yoy)}')
print()
if p_okun < 0.05:
    print('✓ Statistically significant at α = 0.05')
else:
    print('✗ Not statistically significant at α = 0.05')

## 9. Visualisation — YoY Change (Okun's Law)

In [ ]:
yoy_plot = df[['REF_AREA', 'QUARTER', 'GDP_GROWTH_YOY_PCT', 'UNE_CHANGE_YOY_PP', 'GDP_GROUP']].dropna()

colors = yoy_plot['GDP_GROUP'].map({'HIGH': '#1f77b4', 'LOW': '#d62728'})

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(yoy_plot['GDP_GROWTH_YOY_PCT'], yoy_plot['UNE_CHANGE_YOY_PP'],
           c=colors, alpha=0.6, edgecolors='white', linewidth=0.5, s=60)

# OLS trend line
x_vals = yoy_plot['GDP_GROWTH_YOY_PCT'].values
y_vals = yoy_plot['UNE_CHANGE_YOY_PP'].values
slope, intercept, *_ = stats.linregress(x_vals, y_vals)
x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
ax.plot(x_line, slope * x_line + intercept, color='black', linestyle='--', linewidth=1.5, label='OLS trend')

ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
ax.axvline(0, color='grey', linewidth=0.8, linestyle=':')

ax.set_xlabel('GDP per Capita — Year-over-Year Growth (%)', fontsize=12)
ax.set_ylabel('Unemployment Rate — Year-over-Year Change (pp)', fontsize=12)
ax.set_title("Okun's Law — GDP Growth vs Unemployment Change\n(all OECD countries, 2023–2025, quarterly)", fontsize=13)

high_patch = mpatches.Patch(color='#1f77b4', label='High-GDP countries')
low_patch  = mpatches.Patch(color='#d62728', label='Low-GDP countries')
ax.legend(handles=[high_patch, low_patch, plt.Line2D([0],[0],color='black',linestyle='--',label='OLS trend')])

plt.tight_layout()
plt.savefig('datasets/extended/okun_law_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Slope: {slope:.4f}  (Okun coefficient: for each 1% GDP growth, unemployment changes {slope:.3f} pp)')

## 10. Export Featured Dataset

In [ ]:
df.to_csv('datasets/extended/gdp_unemployment_featured.csv', index=False, sep=';')

print('Saved: datasets/extended/gdp_unemployment_featured.csv')
print(f'Final shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')